In [1]:
import os
import pandas as pd
import numpy as np
import re
from pathlib import Path

initialDB_path = Path('../data/initialDB_sequences.csv')
initialDB = pd.read_csv(initialDB_path, sep=',', header=0, comment='#', names=[
    'gene_name', 'gene_id', 'transcript_id', 'protein_id', "5' UTR", 'CDS', "3' UTR", 'protein_sequence', '5\' UTR_len', 'CDS_len', '3\' UTR_len'])

df = initialDB.copy()
df.head()

,gene_name,gene_id,transcript_id,protein_id,5' UTR,CDS,3' UTR,protein_sequence,5' UTR_len,CDS_len,3' UTR_len
0,OR4F5,ENSG00000186092.7,ENST00000641515.2,ENSP00000493376.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,MKKVTAEAISWNESTSETNNSMVTEFIFLGLSDSQELQTFLFMLFF...,60,981,0
1,SAMD11,ENSG00000187634.14,ENST00000968544.1,ENSP00000638603.1,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,509,2448,0
2,SAMD11,ENSG00000187634.14,ENST00000616016.5,ENSP00000478421.2,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,509,2535,0
3,SAMD11,ENSG00000187634.14,ENST00000618323.5,ENSP00000480678.2,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,509,2538,0
4,SAMD11,ENSG00000187634.14,ENST00000968542.1,ENSP00000638601.1,GCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTGG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,508,2256,0


In [2]:
def gc_content(seq):
    if not isinstance(seq, str):
        return np.nan
    seq = seq.strip().upper()
    if not seq:
        return np.nan
    gc = seq.count('G') + seq.count('C')
    return gc / len(seq)

df = initialDB.copy()
df['5_UTR_GC'] = df['5\' UTR'].fillna('').map(gc_content)
df['CDS_GC'] = df['CDS'].fillna('').map(gc_content)
df['3_UTR_GC'] = df['3\' UTR'].fillna('').map(gc_content)
# Ensure all parts are strings before concatenation to avoid: expected str instance, int found
df['global_GC'] = (
    df[["5' UTR", "CDS", "3' UTR"]]
    .fillna('')
    .astype(str)
    .agg(''.join, axis=1)
    .map(gc_content)
)

In [3]:
motifs = {
"ATTTA": r"A[TU]{3}A",
"WTTTW": r"[ACGTU][TU]{3}[ACGTU]",
"WWTTTWW": r"[ACGTU]{2}[TU]{3}[ACGTU]{2}",
"WWWTTTWWW": r"[ACGTU]{3}[TU]{3}[ACGTU]{3}",
"WWWWTTTWWWW": r"[ACGTU]{4}[TU]{3}[ACGTU]{4}",
"WWWWWTTTWWWWW": r"[ACGTU]{5}[TU]{3}[ACGTU]{5}",
"TTTGTTT": r"[TU]{3}G[TU]{3}",
"GTTTG": r"G[TU]{3}G",
"AWTAAA": r"A[ACGTU][TU]AAA"
}

def scan_motifs(df, sequence_col, count_prefix):
    """
    Scans a specific column for motifs and adds a total count column.
    Returns a detailed breakdown DataFrame for those specific motifs.
    """
    total_col_name = f"{count_prefix}_AREs"
    df[total_col_name] = 0

    # Ensure the sequence column is string-like for .str operations
    seq_series = df[sequence_col].fillna('').astype(str)

    motif_counts_dict = {
        'gene_id': df['gene_id'],
        'transcript_id': df['transcript_id'],
        f'{count_prefix}_sequence': seq_series
    }
    
    for name, pattern in motifs.items():
        counts = seq_series.str.findall(pattern, flags=re.IGNORECASE).str.len().fillna(0).astype(int)
        
        df[total_col_name] += counts
        
        motif_counts_dict[f"{count_prefix}_{name}_count"] = counts
    
    detailed_df = pd.DataFrame(motif_counts_dict)
    return df, detailed_df

df, AREs_5_df = scan_motifs(df, '5\' UTR', '5')

df, AREs_3_df = scan_motifs(df, '3\' UTR', '3')

In [4]:
def scan_uorf(df, utr5_col):
    """
    Scans 5' UTR for:
    1. uAUGs: Any AUG in the 5' UTR
    2. uORF: AUG in frame with the canonical start codon 
    """

    def classify_utr_motifs(seq):
        if not isinstance(seq, str):
            return 0, 0

        seq = seq.strip().upper()
        if len(seq) < 3:
            return 0, 0

        # uAUG scanning
        atg_indices = [m.start() for m in re.finditer(r'(?=ATG)', seq)]
        uaug_count = len(atg_indices)

        # uORF scanning
        uorf_count = 0
        seq_len = len(seq)
        for start_idx in atg_indices:
            distance_to_end = seq_len - start_idx
            if distance_to_end % 3 == 0:
                uorf_count += 1

        return uaug_count, uorf_count

    results = df[utr5_col].fillna('').astype(str).apply(classify_utr_motifs)
    df[['uAUG_count', 'uORF_in_frame']] = pd.DataFrame(
        results.tolist(), index=df.index
    )
    return df

def scan_dorf(df, utr3_col):
    """
    Scans 3' UTR for dORFs: 
    Downstream ORFs starting with AUG in the 3' UTR.
    Returns count of dORFs that find a stop codon vs. those that don't (overlap/truncated).
    """

    def classify_dorf_motifs(seq):
        if not isinstance(seq, str):
            return 0, 0

        seq = seq.strip().upper()
        if len(seq) < 3:
            return 0, 0

        atg_indices = [m.start() for m in re.finditer(r'(?=ATG)', seq)]

        non_overlap = 0 # Finding a stop codon within the 3' UTR
        overlap = 0     # Running off the end of the 3' UTR without a stop

        stops = {'TAA', 'TAG', 'TGA'}

        for start_idx in atg_indices:
            remaining_seq = seq[start_idx:]
            found_stop = False

            # Check every 3 bases from the current ATG
            for i in range(0, len(remaining_seq) - 2, 3):
                codon = remaining_seq[i:i+3]
                if codon in stops:
                    found_stop = True
                    break

            if found_stop:
                non_overlap += 1
            else:
                overlap += 1

        return non_overlap, overlap

    results = df[utr3_col].fillna('').astype(str).apply(classify_dorf_motifs)
    df[['dORF_with_stop', 'dORF_truncated']] = pd.DataFrame(
        results.tolist(), index=df.index
    )
    return df

df = scan_uorf(df, '5\' UTR')
df = scan_dorf(df, '3\' UTR')

In [5]:
IRES_path = Path('../data/utr_ire_scan_results.csv')
IRES_db = pd.read_csv(IRES_path, sep=',', header=0, comment='#', names=[
    'gene_id', 'transcript_id', '5_UTR', 'CDS', '3_UTR', '5_UTR_len', 'CDS_len', '3_UTR_len', 'IRES count', 'IRES IDs'
])


cols_to_keep = [col for col in IRES_db.columns if col == 'transcript_id' or col not in df.columns]

df = pd.merge(df, IRES_db[cols_to_keep], on='transcript_id', how='left')

In [6]:
mfe_path = Path('../data/mfe_results.csv')
mfe_db = pd.read_csv(mfe_path, sep=',', header=0, comment='#', names=[
    'gene_id', 'transcript_id', '5_UTR', 'CDS', '3_UTR', '5_len', 'CDS_len', '3_len', '5_UTR_MFE', 'CDS_MFE', '3_UTR_MFE'
])
### Note: MFE values of NaN were not able to be folded by RNAfold either because the sequence is length 0 or it's too long (over 100k nt) for RNAfold to handle.

cols_to_keep = [col for col in mfe_db.columns if col == 'transcript_id' or col not in df.columns]

df = pd.merge(df, mfe_db[cols_to_keep], on='transcript_id', how='left')

In [7]:
# #translation efficiency predictions from RiboNN

# RiboNN_prediction_path = Path('../data/prediction_output.csv')
# RiboNN_predictions = pd.read_csv(RiboNN_prediction_path, sep=',', header=0, comment='#', names=[
#     'transcript_id', '5\' UTR', 'CDS', '3\' UTR', 'predicted_TE_108T', 'predicted_TE_12T', 'predicted_TE_A2780', 
#     'predicted_TE_A549', 'predicted_TE_BJ', 'predicted_TE_BRx.142', 'predicted_TE_C643', 'predicted_TE_CRL.1634', 'predicted_TE_Calu.3', 
#     'predicted_TE_Cybrid_Cells', 'predicted_TE_H1.hESC', 'predicted_TE_H1933', 'predicted_TE_H9.hESC', 'predicted_TE_HAP.1', 
#     'predicted_TE_HCC_tumor', 'predicted_TE_HCC_adjancent_normal', 'predicted_TE_HCT116', 'predicted_TE_HEK293', 'predicted_TE_HEK293T', 
#     'predicted_TE_HMECs', 'predicted_TE_HSB2', 'predicted_TE_HSPCs', 'predicted_TE_HeLa', 'predicted_TE_HeLa_S3', 'predicted_TE_HepG2', 
#     'predicted_TE_Huh.7.5', 'predicted_TE_Huh7', 'predicted_TE_K562', ' predicted TE_Kidney_normal_tissue', ' predicted TE_LCL', 
#     ' predicted TE_LuCaP.PDX', ' predicted TE_MCF10A', ' predicted TE_MCF10A.ER.Src', ' predicted TE_MCF7', ' predicted TE_MD55A3', 
#     ' predicted TE_MDA.MB.231', ' predicted TE_MM1.S', ' predicted TE_MOLM.13',' predicted TE_Molt.3', ' predicted TE_Mutu', 
#     ' predicted TE_OSCC', ' predicted TE_PANC1', ' predicted TE_PATU.8902', ' predicted TE_PC3', ' predicted TE_PC9', 
#     ' predicted TE_Primary_CD4._T.cells', ' predicted TE_Primary_human_bronchial_epithelial_cells', ' predicted TE_RD.CCL.136', 
#     ' predicted TE_RPE.1', ' predicted TE_SH.SY5Y', ' predicted TE_SUM159PT', ' predicted TE_SW480TetOnAPC', ' predicted TE_T47D', 
#     ' predicted TE_THP.1', ' predicted TE_U.251', ' predicted TE_U.343', ' predicted TE_U2392', ' predicted TE_U2OS', ' predicted TE_Vero_6', 
#     ' predicted TE_WI38', '_predicted_TE_WM902B', '_predicted_TE_WTC.11', '_predicted_TE_ZR75.1', '_predicted_TE_cardiac_fibroblasts', 
#     '_predicted_TE_ccRCC', '_predicted_TE_early_neurons', '_predicted_TE_fibroblast', '_predicted_TE_hESC', '_predictedTE_human_brain_tumor', 
#     '_predictedTE_iPSC.differentiated_dopamine_neurons', '_predictedTE_megakaryocytes', '_predictedTE_muscle_tissue', '_predictedTE_neuronal_precursor_cells',
#     '_predictedTE_neurons', '_predictedTE_normal_brain_tissue', '_predictedTE_normal_prostate', '_predictedTE_primary_macrophages',
#     '_predictedTE_skeletal_muscle','mean_predicted_TE'
# ])

# cols_to_keep = [col for col in RiboNN_predictions.columns if col == 'transcript_id' or col not in df.columns]

# df = pd.merge(df, RiboNN_predictions[cols_to_keep], on='transcript_id', how='left')

In [8]:
import numpy as np
import re

# Classic Sharp & Li (1987) E. coli index 
SHARP_ECOLI_INDEX = {
    'TTT': 0.10, 'TTC': 1.00, 'TTA': 0.02, 'TTG': 0.02, 'CTT': 0.04, 'CTC': 0.03, 'CTA': 0.01, 'CTG': 1.00,
    'ATT': 0.17, 'ATC': 1.00, 'ATA': 0.003, 'ATG': 1.00, 'GTT': 1.00, 'GTC': 0.06, 'GTA': 0.49, 'GTG': 0.13,
    'TCT': 1.00, 'TCC': 0.74, 'TCA': 0.07, 'TCG': 0.01, 'CCT': 0.07, 'CCC': 0.01, 'CCA': 1.00, 'CCG': 0.15,
    'ACT': 1.00, 'ACC': 1.00, 'ACA': 0.07, 'ACG': 0.01, 'GCT': 1.00, 'GCC': 0.13, 'GCA': 0.58, 'GCG': 0.17,
    'TAT': 0.24, 'TAC': 1.00, 'CAA': 0.12, 'CAG': 1.00, 'AAT': 0.05, 'AAC': 1.00, 'GAT': 0.43, 'GAC': 1.00,
    'GAA': 1.00, 'GAG': 0.22, 'TGT': 0.50, 'TGC': 1.00, 'TGG': 1.00, 'CGT': 1.00, 'CGC': 0.36, 'CGA': 0.004,
    'CGG': 0.01, 'AGA': 0.004, 'AGG': 0.002, 'GGT': 1.00, 'GGC': 0.72, 'GGA': 0.01, 'GGG': 0.01, 'AAA': 1.00, 
    'AAG': 0.25, 'CCT': 0.07, 'CCC': 0.01, 'CCA': 1.00, 'CCG': 0.15
}

class CodonAdaptationIndex:
    def __init__(self, index_dict):
        self._cai_index = {k.upper(): float(v) for k, v in index_dict.items()}

    def cai_for_gene(self, seq):
        if not isinstance(seq, str):
            return np.nan

        # Clean sequence strings
        seq = re.sub(r"\s+", "", seq).upper().replace("U", "T")
        codons = [seq[i:i+3] for i in range(0, len(seq) - 2, 3)]

        weights = [self._cai_index[c] for c in codons if c in self._cai_index]
        if not weights:
            return np.nan

        # Guard against log(0)
        weights = [w if w > 0 else 1e-12 for w in weights]
        return float(np.exp(np.mean(np.log(weights))))

# Initialize with the hardcoded index
cai_calc = CodonAdaptationIndex(SHARP_ECOLI_INDEX)

# Apply directly to your DataFrame
df['CAI'] = df['CDS'].apply(cai_calc.cai_for_gene)
df.head()

,gene_name,gene_id,transcript_id,protein_id,5' UTR,CDS,3' UTR,protein_sequence,5' UTR_len,CDS_len,...,5_UTR_len,3_UTR_len,IRES count,IRES IDs,5_len,3_len,5_UTR_MFE,CDS_MFE,3_UTR_MFE,CAI
0,OR4F5,ENSG00000186092.7,ENST00000641515.2,ENSP00000493376.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,MKKVTAEAISWNESTSETNNSMVTEFIFLGLSDSQELQTFLFMLFF...,60,981,...,60.0,0.0,0.0,NaN,60.0,0.0,-9.5,-260.2,NaN,0.205816
1,SAMD11,ENSG00000187634.14,ENST00000968544.1,ENSP00000638603.1,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,509,2448,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.187306
2,SAMD11,ENSG00000187634.14,ENST00000616016.5,ENSP00000478421.2,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,509,2535,...,509.0,0.0,0.0,NaN,509.0,0.0,-321.0,-1251.6,NaN,0.186715
3,SAMD11,ENSG00000187634.14,ENST00000618323.5,ENSP00000480678.2,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,509,2538,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.186982
4,SAMD11,ENSG00000187634.14,ENST00000968542.1,ENSP00000638601.1,GCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTGG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,508,2256,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.190319


In [9]:
df.to_csv('../data/expandedDB.csv', index=False)